In [ ]:
import dask
import time
import toolviper
import random
import webbrowser
import pathlib

# ====== Testing ====== #

import asyncio

# ===================== #


import toolviper.utils.logger as logger

import toolviper.dask.client as client

from xradio.measurement_set import convert_msv2_to_processing_set
from xradio.measurement_set import open_processing_set

In [ ]:
# def pipeline_a(x):
#     import time
#     time.sleep(10)  # Simulating heavy work
#     return x * 2

# def pipeline_b(x):
#     import time
#     time.sleep(10)
#     return x + 100

# async def manage_jobs():
#     async with client.local_client(
#         cores=12,
#         asynchronous=True,
#         log_params={
#             "log_to_file":False,
#             "log_to_term":True,
#             "log_level":"DEBUG"
#         },
#         worker_log_params={
#             "log_to_file":False,
#             "log_to_term":True,
#             "log_level":"INFO"
#         }
#     )as c:
#         logger.info(c.dashboard_link)
#         logger.info("Submitting both graphs to the cluster...")

#         # 2. Submit graph 1 (runs in the background)
#         future_a = c.submit(pipeline_a, 10)

#         # 3. Submit graph 2 (runs in the background at the same time)
#         future_b = c.submit(pipeline_b, 10)

#         # 4. Use asyncio.gather to await both concurrently
#         # client.gather is a coroutine when client is asynchronous
#         results = await asyncio.gather(
#             c.gather(future_a),
#             c.gather(future_b)
#         )

# async def submit():
#     await manage_jobs()

In [ ]:
# result = await submit()

In [ ]:
client = client.local_client(
    cores=12,
    asynchronous=False,
    log_params={
        "log_to_file": False, 
        "log_to_term": True, 
        "log_level": "DEBUG"
        },
    
    worker_log_params={
        "log_to_file": False, 
        "log_to_term": True, 
        "log_level": 
        "INFO"
        },
)

In [ ]:
client.dashboard_link

#### Utilities

In [ ]:
def generate_delay(n=1, m=2):
    time.sleep(random.uniform(n, m))


def args_checker(args):
    if "delay" in args[0].keys():
        if args[0]["task_coords"]["antenna_name"]["data"] == args[0]["delay"]:
            return True

    return False


def write_image(file_name, spw, antenna, field):

    path = pathlib.Path(file_name)
    logger.info(str(path))
    if not path.exists():
        logger.error(f"File: {str(path)} not found.")

    data_path = path.joinpath(spw).joinpath(field).joinpath(antenna)
    logger.info(str(data_path))
    data_path.mkdir(exist_ok=True, parents=True)

    with open(
        str(data_path.joinpath(f"some_data_{spw}.{field}.{antenna}.image.ps.zarr")), "w"
    ) as file:
        file.write(f"{spw}\t{field}\t{antenna}")

### Make Test Functions 

In [ ]:
UPPER_DELAY_LIMIT = 2


# Distributed #[field, spw, antenna][EB]
def deviation_mask_heuristic(*args, **kwargs):
    logger.info("Deviation mask heuristic ... [field, spw, antenna]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)


# Distributed #[field,spw][EB]
def line_analysis(*args, **kwargs):
    logger.info("Line analysis... [field, spw]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)


# Distributed #[field,spw, antenna, pol][EB]
def fitting_heuristic(*args, **kwargs):
    logger.info("Fitting Heuristic ... [field, spw, antenna, pol]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)


# Distributed #[field,spw, antenna, pol][EB]
def baseline_subtraction(*args, **kwargs):
    logger.info("Baseline subtraction ... [field, spw, antenna, pol]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)


# Distributed #[field,spw, antenna, pol][EB]
def heuristic_plots_per_eb(*args, **kwargs):
    logger.info("Pre-EB heuristic plotting ... [field, spw, antenna, pol]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)


# Distributed #[field, spw]
def heuristic_plots(*args, **kwargs):
    logger.info("Heuristic plotting ... [field, spw]")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

# Standard Gather
def gather(*args, **kwargs):
    logger.info(" Gathering ...")
    generate_delay(n=1, m=UPPER_DELAY_LIMIT)

### Convert the MSv2 to a Processing Set and Open

In [ ]:
pathlib.Path().cwd()

In [ ]:
processing_set = str(
    "/Users/joshua/Development/processing_set/uid___A002_Xcd07af_X70c7.2spws.80000chans.lsrk.ps.zarr"
)

ps = open_processing_set(
    ps_store=str(processing_set), scan_intents=["OBSERVE_TARGET#ON_SOURCE"]
)

In [ ]:
ps["uid___A002_Xcd07af_X70c7.2spws.80000chans.lsrk_0"].xr_ms.get_partition_info()

### Demonstrate the Simple Node Building Tool

In [ ]:
graph = toolviper.utils.sd.graph.Graph.from_dataset(ps)

In [ ]:
graph.filter(leaves=["uid___A002_Xcd07af_X70c7.2spws.80000chans.lsrk_0", "uid___A002_Xcd07af_X70c7.2spws.80000chans.lsrk_1"])

In [ ]:
graph.datatree

In [ ]:
graph.make_coordinates(coords=["field_name", "antenna_name", "polarization"])

In [ ]:
graph.coordinates

In [ ]:
graph.build_node(ps_partition=["field_name", "antenna_name", "polarization"])

In [ ]:
graph.node_mapping

In [ ]:
graph.map(
    function=line_analysis, parameters={"parameter": False}, connect=None
)  # Adding make_workflow=True here will allow you to produce results without a gather.

graph.reduce(parameters={"parameter": False}, function=fitting_heuristic, mode="tree")

In [ ]:
graph.node_mapping

In [ ]:
dask.visualize(graph._graph, filename="map_graph")

In [ ]:
graph.reset()

### Example of `hsd_baseline()` using GraphViper Framework

In [ ]:
# Spawn dashboard window in a separate tab,
# comment out if you don't want this to spawn.
webbrowser.open(url=client.dashboard_link)

def hsd_baseline(processing_set, leaves=None):

    # Begin Preprocessing
    ps = open_processing_set(
        ps_store=str(processing_set), scan_intents=["OBSERVE_TARGET#ON_SOURCE"]
    )
    
    job = toolviper.utils.sd.graph.Graph.from_dataset(ps)

    # If you want to filter sub-dataset
    if leaves:
        job.filter(leaves=leaves)
    
    # Processing Block 1 (spw comes for free) [spw, field, antenna]
    job.make_coordinates(coords=["field_name", "antenna_name"])
    job.build_node(ps_partition=["field_name", "antenna_name"])
    
    job.map(
        function=deviation_mask_heuristic,
        parameters={}
    )

    job.reduce(
        function=gather,
        mode="single_node",
    )

    job.compute()
    
    
    # Processing Block 2 (spw comes for free) [spw, field]
    job.make_coordinates(coords=["field_name"])
    job.build_node(ps_partition=["field_name"])
    
    job.map(
        function=line_analysis,
        parameters={}
    )

    job.reduce(
        function=gather,
        mode="single_node",
    )

    # Finish Dask processing - compute job graph
    job.compute()

    # Processing Block 3 (spw comes for free) [spw, field, antenna, pol]
    job.make_coordinates(coords=["field_name", "antenna_name", "polarization"])
    job.build_node(ps_partition=["field_name", "antenna_name", "polarization"])
    
    job.map(
        function=fitting_heuristic,
        parameters={}
    )

    job.reduce(
        function=gather,
        mode="single_node",
    )

    job.compute()

    # Processing Block 4 (spw comes for free) [spw, field, antenna, pol]
    job.make_coordinates(coords=["field_name", "antenna_name", "polarization"])
    job.build_node(ps_partition=["field_name", "antenna_name", "polarization"])
    
    job.map(
        function=baseline_subtraction,
        parameters={}
    )

    job.reduce(
        function=gather,
        mode="single_node",
    )

    job.compute()

    # Processing Block 5 (spw comes for free) [spw, field, antenna, pol]
    job.make_coordinates(coords=["field_name", "antenna_name", "polarization"])
    job.build_node(ps_partition=["field_name", "antenna_name", "polarization"])
    
    job.map(
        function=heuristic_plots_per_eb,
        parameters={}
    )

    job.reduce(
        function=gather,
        mode="single_node",
    )

    job.compute()

    # Processing Block 6 (spw comes for free) [spw, field]
    job.make_coordinates(coords=["field_name"])
    job.build_node(ps_partition=["field_name"])
    
    job.map(
        function=heuristic_plots,
        parameters={},
        make_workflow=True
    )

    job.compute()

    return job

In [ ]:
processing_set = str(
    "/Users/joshua/Development/processing_set/uid___A002_Xcd07af_X70c7.2spws.80000chans.lsrk.ps.zarr"
)

job = hsd_baseline(
    processing_set=processing_set,
    leaves=[
       "uid___A002_Xcd07af_X70c7.2spws.80000chans.lsrk_0", 
       "uid___A002_Xcd07af_X70c7.2spws.80000chans.lsrk_1"
    ],
)